# tailbench — compare runs

Judges whether a change to `program.rs` did anything. For characterising a single
run, use `analyse_run.ipynb`.

Three things to separate, because all three look like "the number went down":

1. **A real architectural win** — tail falls, `ok%` holds, queue wait drops.
2. **A win bought by failing requests** — tail falls, `ok%` falls with it.
3. **Noise** — a change inside replay variance. Use `--repeat N` and compare the
   delta against the reported std. dev.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd() if (pathlib.Path.cwd()/"tailbench_viz.py").exists()
                      else pathlib.Path.cwd()/"notebooks"))

import matplotlib.pyplot as plt
import pandas as pd
import tailbench_viz as tv

plt.rcParams.update({"figure.dpi": 110, "font.size": 9,
                     "axes.titlesize": 11, "figure.facecolor": "white"})
pd.set_option("display.width", 200, "display.max_columns", 50)

## 1. Pick a baseline and a candidate

Ideally identical scenario and seed, differing only in `program.rs` — otherwise the
delta includes sampling noise as well as the change.

In [ ]:
available = tv.load_runs("../results")
tv.compare(available)

In [ ]:
BASELINE  = available[-2].path
CANDIDATE = available[-1].path

base, cand = tv.load_run(BASELINE), tv.load_run(CANDIDATE)

if base.manifest["seed"] != cand.manifest["seed"]:
    print("WARNING: different seeds -- the delta includes sampling noise")
if base.scenario_id != cand.scenario_id:
    print(f"WARNING: different scenarios ({base.scenario_id} vs {cand.scenario_id}) -- not comparable")
for r in (base, cand):
    if not r.authoritative:
        print(f"note: {r.label} is {r.manifest['environment']} -- relative comparison only")

## 2. The delta

`direction` is per-metric, not just the sign: a falling `ok_rate` is a regression even
though the number went down. Tail improving while `ok_rate` falls is case 2 above —
latency bought by failing requests.

In [ ]:
tv.delta_table(base, cand).round(4)

## 3. Tails side by side

**Left:** both tail CDFs on one axis. Curves that separate *only in the far tail* are
the interesting case — that is the regime `p99` cannot see.

**Right:** headline metrics. `p99` and `cvar_99` are deliberately adjacent: when an
architectural change moves `cvar_99` far more than `p99`, the two bars diverging *is*
the finding.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.4))
tv.plot_compare_tails([base, cand], axes[0])
tv.plot_metric_bars([base, cand], axes[1])
fig.tight_layout()

b, cr = base.report, cand.report
dp99  = 100 * (cr["p99"] - b["p99"]) / b["p99"]
dcvar = 100 * (cr["cvar_99"] - b["cvar_99"]) / b["cvar_99"]
dok   = 100 * (cr["ok_rate"] - b["ok_rate"])
print(f"p99      {b['p99']:7.2f} -> {cr['p99']:7.2f} ms  ({dp99:+6.2f}%)")
print(f"cvar_99  {b['cvar_99']:7.2f} -> {cr['cvar_99']:7.2f} ms  ({dcvar:+6.2f}%)")
print(f"ok       {100*b['ok_rate']:7.3f} -> {100*cr['ok_rate']:7.3f} %   ({dok:+6.3f} pp)")

if abs(dcvar) > 2 * abs(dp99):
    print(f"\ncvar_99 moved {abs(dcvar)/max(abs(dp99),1e-9):.1f}x more than p99 -- a change in the")
    print("*shape* of the tail. p99 alone would have understated it.")
if dcvar < 0 and dok < 0:
    print("\nWARNING: tail improved while ok_rate fell. Check whether the gain was")
    print("bought by failing requests rather than by serving them faster.")

## 4. Where the difference came from

The delta says *how much*; these say *why*. Queue wait is the program's own doing, so
a change that moved the tail should show up here — if it does not, the tail moved for
some other reason.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8.5))
for ax_row, r, name in [(axes[0], base, "baseline"), (axes[1], cand, "candidate")]:
    tv.plot_downstreams(r, ax_row[0])
    tv.plot_by_class(r, ax_row[1])
    ax_row[0].set_title(f"{name}: downstream service vs queue", fontsize=10, loc="left")
    ax_row[1].set_title(f"{name}: latency by class", fontsize=10, loc="left")
fig.tight_layout()

In [ ]:
# Queue wait per downstream, both runs. Service time is fixed by the scenario, so
# any real difference here is the program's doing.
rows = []
for r, name in [(base, "baseline"), (cand, "candidate")]:
    sp = r.spans()
    sp = sp[sp.post_warmup]
    for ds, g in sp.groupby("downstream_id"):
        rows.append({"run": name, "downstream": ds, "calls": len(g),
                     "queue_p99": tv.percentile(g.queue_wait_ms, .99),
                     "service_p99": tv.percentile(g.service_ms, .99),
                     "total_p99": tv.percentile(g.total_ms, .99)})
if rows:
    display(pd.DataFrame(rows).pivot(index="downstream", columns="run").round(3))

## 5. Sweeping a series

A baseline plus successive attempts, all on one axis. This is the view for a sequence
of LLM edits rather than a single A/B.

In [ ]:
SCENARIO = cand.scenario_id
sweep = [r for r in available if r.scenario_id == SCENARIO]

if len(sweep) < 2:
    print(f"only {len(sweep)} run(s) of {SCENARIO}; run the scenario again to compare")
else:
    display(tv.compare(sweep).round(3))
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.4))
    tv.plot_compare_tails(sweep, axes[0])
    tv.plot_metric_bars(sweep, axes[1])
    fig.tight_layout()

## 6. Is the delta bigger than the noise?

A delta smaller than replay std. dev. is not a result. Get the denominator with:

```bash
scripts/run.sh scenarios/level3.toml --repeat 5
```

which reports the std. dev. of `cvar_99` and `p99` across replays. The cell below
computes it directly if either run is a `--repeat` directory.

In [ ]:
for r, name in [(base, "baseline"), (cand, "candidate")]:
    reps = sorted(r.df.replay.unique())
    if len(reps) < 2:
        print(f"{name}: single run, no replay noise to measure")
        continue
    vals = [(tv.cvar(g[g.post_warmup].scored_ms, .99),
             tv.percentile(g[g.post_warmup].scored_ms, .99))
            for _, g in r.df.groupby("replay")]
    cv = pd.Series([v[0] for v in vals]); p9 = pd.Series([v[1] for v in vals])
    print(f"{name}: {len(reps)} replays")
    print(f"  cvar_99  mean {cv.mean():.2f}  sd {cv.std():.3f} ms")
    print(f"  p99      mean {p9.mean():.2f}  sd {p9.std():.3f} ms")